In [1]:
from ucimlrepo import fetch_ucirepo

auto_mpg = fetch_ucirepo(id=9)

X = auto_mpg.data.features
y = auto_mpg.data.targets.squeeze()

In [3]:
X.head()

,displacement,cylinders,horsepower,weight,acceleration,model_year,origin
0,307.0,8,130.0,3504,12.0,70,1
1,350.0,8,165.0,3693,11.5,70,1
2,318.0,8,150.0,3436,11.0,70,1
3,304.0,8,150.0,3433,12.0,70,1
4,302.0,8,140.0,3449,10.5,70,1


In [4]:
X.isnull().any()

displacement    False
cylinders       False
horsepower       True
weight          False
acceleration    False
model_year      False
origin          False
dtype: bool

In [5]:
import numpy as np 
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression,Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

In [6]:
X_train,X_test,y_train,y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [7]:
lin_model = Pipeline([
    ("imputer", SimpleImputer()),
    ("scaler",StandardScaler()),
    ("model",LinearRegression())
    
])

In [9]:
lin_model.fit(X_train,y_train)

Pipeline(steps=[('imputer', SimpleImputer()), ('scaler', StandardScaler()),
                ('model', LinearRegression())])

In [10]:
y_train_pred = lin_model.predict(X_train)
y_test_pred = lin_model.predict(X_test)

In [14]:
print("linear regression")
print("Train R²:", lin_model.score(X_train, y_train))
print("Test R² :", lin_model.score(X_test, y_test))

linear regression
Train R²: 0.8139609620580498
Test R² : 0.8475695460782322


In [15]:
for alpha in [0.01, 0.1, 1, 10, 100, 1000]:

    ridge_model = Pipeline([
        ("imputer", SimpleImputer()),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=alpha))
    ])

    ridge_model.fit(X_train, y_train)

    train_score = ridge_model.score(X_train, y_train)
    test_score = ridge_model.score(X_test, y_test)
    print("Ridge")
    print(
        "Alpha:", alpha,
        "| Train R²:", round(train_score, 3),
        "| Test R²:", round(test_score, 3)
    )

Ridge
Alpha: 0.01 | Train R²: 0.814 | Test R²: 0.848
Ridge
Alpha: 0.1 | Train R²: 0.814 | Test R²: 0.848
Ridge
Alpha: 1 | Train R²: 0.814 | Test R²: 0.847
Ridge
Alpha: 10 | Train R²: 0.811 | Test R²: 0.846
Ridge
Alpha: 100 | Train R²: 0.785 | Test R²: 0.839
Ridge
Alpha: 1000 | Train R²: 0.62 | Test R²: 0.704


In [16]:
for alpha in [0.01, 0.1, 1]:

    ridge_model = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=alpha))
    ])

    ridge_model.fit(X_train, y_train)

    print(
        alpha,
        ridge_model.score(X_test, y_test)
    )

0.01 0.847521145458213
0.1 0.8475051720147506
1 0.8473369652391096


# Ridge Regression — Final Notes

## What is Ridge Regression?

Ridge Regression is **Linear Regression with L2 Regularization**.

It helps control large or unstable coefficients and is especially useful when features are highly correlated.

---

## Main Idea

Linear Regression tries to minimize prediction error.

Ridge Regression tries to:

> **Make good predictions while keeping coefficients under control.**

[
\text{Ridge Loss} = \text{MSE} + \alpha \sum \beta^2
]

* **MSE** → prediction error
* **β** → coefficients
* **α (alpha)** → regularization strength
* **Σβ²** → penalty for large coefficients

---

## Understanding Alpha

| Alpha         | Effect                     |
| ------------- | -------------------------- |
| `α ≈ 0`       | Almost Linear Regression   |
| Small `α`     | Weak regularization        |
| Medium `α`    | More coefficient shrinkage |
| Large `α`     | Strong regularization      |
| Too large `α` | Can cause underfitting     |

### Simple Rule

```text
Alpha ↑
   ↓
Penalty ↑
   ↓
Coefficients are more constrained
   ↓
Model flexibility ↓
   ↓
Too much → Underfitting
```

---

## Why Use StandardScaler?

Ridge penalizes coefficients, so feature scale matters.

Therefore, a common workflow is:

```text
X_train
   ↓
StandardScaler
   ↓
Ridge
   ↓
Prediction
```

Using a Pipeline:

```python
ridge_model = Pipeline([
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])
```

If missing values exist:

```python
ridge_model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("model", Ridge(alpha=1.0))
])
```

---

## Ridge vs Linear Regression

### Linear Regression

* No regularization
* Coefficients can become large or unstable
* Can struggle with multicollinearity
* May overfit in some situations

### Ridge Regression

* Uses **L2 regularization**
* Penalizes large coefficients
* Tends to shrink the coefficient vector
* Handles multicollinearity better
* Generally keeps all features
* Can improve model stability and generalization

---

## Important

Ridge does **not**:

* Remove correlation between features
* Normally set coefficients exactly to zero
* Automatically improve R²
* Automatically make a model faster or lighter

Also, not every individual Ridge coefficient must become smaller than its Linear Regression counterpart. With correlated features, Ridge can redistribute coefficient values while reducing the overall regularized coefficient magnitude.

---

## When Should I Use Ridge?

Consider Ridge when:

* Linear Regression is overfitting
* Features are highly correlated
* Coefficients are large or unstable
* There are many features
* You want regularization without removing features

---

## Ridge vs Polynomial Degree

```text
Polynomial degree ↑
→ Complexity ↑
→ Overfitting risk ↑

Ridge alpha ↑
→ Model flexibility ↓
→ Too much can cause underfitting
```

---

## Key Interview Answer

> **Ridge Regression is Linear Regression with L2 regularization. It penalizes large coefficients, which can make the model more stable when features are correlated and can reduce overfitting. Alpha controls the strength of regularization, and Ridge generally retains all features rather than setting coefficients exactly to zero.**

---

## Key Takeaway

```text
Linear Regression
        ↓
Large / unstable coefficients?
        ↓
Ridge Regression
        ↓
L2 Regularization
        ↓
Shrink / stabilize coefficients
        ↓
Potentially better generalization
```

**Remember:**

> Ridge is not about getting a higher R² every time. It is about **controlling coefficients and improving model stability/generalization when regularization is needed.**
